In [2]:
!pip install tensorflow

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [2]:
df = pd.read_excel("BIA_METAR_DATA_(2019_2024) (1).xlsx")
df.head()

,Year,Month,Date,METAR /SPECI,Time(UTC),Wind Dir.,Wind speed(Kts),Visibility,Weather,Clouds,Dry tem(0C),Dew point(0C),RH(%),QNH (hPa),Unnamed: 14
0,2019,1,1,M,10,30,7,9000,NaN,SKC,22.5,20.8,88,1015.2,NaN
1,2019,1,1,M,40,30,7,9000,NaN,SKC,22.6,20.5,88,1015.5,NaN
2,2019,1,1,M,110,30,8,8000,NaN,SKC,22.8,20.6,87,1015.6,NaN
3,2019,1,1,M,140,30,9,8000,NaN,SKC,23.1,20.7,86,1015.9,NaN
4,2019,1,1,M,210,30,11,9000,NaN,NSC,23.9,20.7,81,1016.3,NaN


In [3]:
df["time_clean"] = (
    df["Time(UTC)"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(4)
)

df["datetime"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str).str.zfill(2) + "-" +
    df["Date"].astype(str).str.zfill(2) + " " +
    df["time_clean"],
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

df = df.dropna(subset=["datetime"])
df = df.sort_values("datetime")

df.head()

,Year,Month,Date,METAR /SPECI,Time(UTC),Wind Dir.,Wind speed(Kts),Visibility,Weather,Clouds,Dry tem(0C),Dew point(0C),RH(%),QNH (hPa),Unnamed: 14,time_clean,datetime
0,2019,1,1,M,10,30,7,9000,NaN,SKC,22.5,20.8,88,1015.2,NaN,0010,2019-01-01 00:10:00
1,2019,1,1,M,40,30,7,9000,NaN,SKC,22.6,20.5,88,1015.5,NaN,0040,2019-01-01 00:40:00
2,2019,1,1,M,110,30,8,8000,NaN,SKC,22.8,20.6,87,1015.6,NaN,0110,2019-01-01 01:10:00
3,2019,1,1,M,140,30,9,8000,NaN,SKC,23.1,20.7,86,1015.9,NaN,0140,2019-01-01 01:40:00
4,2019,1,1,M,210,30,11,9000,NaN,NSC,23.9,20.7,81,1016.3,NaN,0210,2019-01-01 02:10:00


In [4]:
df_ne = df[df["Month"].isin([12, 1, 2])]
df_ne.head()

,Year,Month,Date,METAR /SPECI,Time(UTC),Wind Dir.,Wind speed(Kts),Visibility,Weather,Clouds,Dry tem(0C),Dew point(0C),RH(%),QNH (hPa),Unnamed: 14,time_clean,datetime
0,2019,1,1,M,10,30,7,9000,NaN,SKC,22.5,20.8,88,1015.2,NaN,0010,2019-01-01 00:10:00
1,2019,1,1,M,40,30,7,9000,NaN,SKC,22.6,20.5,88,1015.5,NaN,0040,2019-01-01 00:40:00
2,2019,1,1,M,110,30,8,8000,NaN,SKC,22.8,20.6,87,1015.6,NaN,0110,2019-01-01 01:10:00
3,2019,1,1,M,140,30,9,8000,NaN,SKC,23.1,20.7,86,1015.9,NaN,0140,2019-01-01 01:40:00
4,2019,1,1,M,210,30,11,9000,NaN,NSC,23.9,20.7,81,1016.3,NaN,0210,2019-01-01 02:10:00


In [5]:
df_model = df_ne[[
    "datetime",
    "Dry tem(0C)",
    "RH(%)",
    "QNH (hPa)"
]]

df_model = df_model.rename(columns={
    "Dry tem(0C)": "temperature",
    "RH(%)": "humidity",
    "QNH (hPa)": "pressure"
})

df_model.head()

,datetime,temperature,humidity,pressure
0,2019-01-01 00:10:00,22.5,88,1015.2
1,2019-01-01 00:40:00,22.6,88,1015.5
2,2019-01-01 01:10:00,22.8,87,1015.6
3,2019-01-01 01:40:00,23.1,86,1015.9
4,2019-01-01 02:10:00,23.9,81,1016.3


In [6]:
df_model["temperature"] = pd.to_numeric(df_model["temperature"], errors="coerce")
df_model["humidity"] = pd.to_numeric(df_model["humidity"], errors="coerce")
df_model["pressure"] = pd.to_numeric(df_model["pressure"], errors="coerce")

df_model = df_model.dropna()

df_model.head()

,datetime,temperature,humidity,pressure
0,2019-01-01 00:10:00,22.5,88.0,1015.2
1,2019-01-01 00:40:00,22.6,88.0,1015.5
2,2019-01-01 01:10:00,22.8,87.0,1015.6
3,2019-01-01 01:40:00,23.1,86.0,1015.9
4,2019-01-01 02:10:00,23.9,81.0,1016.3


In [7]:
df_model["temp_next"] = df_model["temperature"].shift(-3)
df_model["humidity_next"] = df_model["humidity"].shift(-3)
df_model["pressure_next"] = df_model["pressure"].shift(-3)

df_model = df_model.dropna()

df_model.head()

,datetime,temperature,humidity,pressure,temp_next,humidity_next,pressure_next
0,2019-01-01 00:10:00,22.5,88.0,1015.2,23.1,86.0,1015.9
1,2019-01-01 00:40:00,22.6,88.0,1015.5,23.9,81.0,1016.3
2,2019-01-01 01:10:00,22.8,87.0,1015.6,24.8,77.0,1016.6
3,2019-01-01 01:40:00,23.1,86.0,1015.9,25.1,74.0,1016.7
4,2019-01-01 02:10:00,23.9,81.0,1016.3,26.6,68.0,1016.6


In [8]:
X = df_model[["temperature", "humidity", "pressure"]]

y = df_model[[
    "temp_next",
    "humidity_next",
    "pressure_next"
]]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [10]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression MAE:", lr_mae)
print("Linear Regression R2:", lr_r2)

Linear Regression MAE: 2.562273053373192
Linear Regression R2: 0.5960549237317855


In [11]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest R2:", rf_r2)

Random Forest MAE: 2.22094450130635
Random Forest R2: 0.46761964924259386


In [12]:
X_train_lstm = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm_model = Sequential()
lstm_model.add(LSTM(50, input_shape=(1, 3)))
lstm_model.add(Dense(3))

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.fit(
    X_train_lstm,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

lstm_pred = lstm_model.predict(X_test_lstm)

lstm_mae = mean_absolute_error(y_test, lstm_pred)
lstm_r2 = r2_score(y_test, lstm_pred)

print("LSTM MAE:", lstm_mae)
print("LSTM R2:", lstm_r2)

Epoch 1/20


C:\Users\imash\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 340961.2812
Epoch 2/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 336494.7500
Epoch 3/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 332135.7188
Epoch 4/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 327857.4062
Epoch 5/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 323648.6875
Epoch 6/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 319501.1562
Epoch 7/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 315408.1875
Epoch 8/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 311365.2188
Epoch 9/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 307367.9062
Epoch 10/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 303415.0938
Epoch 11/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 299505.1562
Epoch 12/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 295636.5312
Epoch 13/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 291808.3125
Epoch 14/20
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 288019.531

In [13]:
ensemble_pred = (
    lr_pred +
    rf_pred +
    lstm_pred
) / 3

ensemble_mae = mean_absolute_error(y_test, ensemble_pred)
ensemble_r2 = r2_score(y_test, ensemble_pred)

print("Hybrid Ensemble MAE:", ensemble_mae)
print("Hybrid Ensemble R2:", ensemble_r2)

Hybrid Ensemble MAE: 101.08633641721597
Hybrid Ensemble R2: -7111.521095026535


In [14]:
results = pd.DataFrame({
    "Model": [
        "Multiple Linear Regression",
        "Random Forest",
        "LSTM",
        "Hybrid Ensemble"
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        lstm_mae,
        ensemble_mae
    ],
    "R2 Score": [
        lr_r2,
        rf_r2,
        lstm_r2,
        ensemble_r2
    ]
})

results

,Model,MAE,R2 Score
0,Multiple Linear Regression,2.562273,0.596055
1,Random Forest,2.220945,0.467620
2,LSTM,300.347198,-64030.921875
3,Hybrid Ensemble,101.086336,-7111.521095


In [15]:
sample = pd.DataFrame([{
    "temperature": 28,
    "humidity": 80,
    "pressure": 1010
}])

lr_output = lr_model.predict(sample)
rf_output = rf_model.predict(sample)

sample_lstm = sample.values.reshape((sample.shape[0], 1, sample.shape[1]))
lstm_output = lstm_model.predict(sample_lstm)

final_output = (lr_output + rf_output + lstm_output) / 3

print("Predicted Temperature next 3h:", round(final_output[0][0], 2))
print("Predicted Humidity next 3h:", round(final_output[0][1], 2))
print("Predicted Pressure next 3h:", round(final_output[0][2], 2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
Predicted Temperature next 3h: 27.27
Predicted Humidity next 3h: 81.37
Predicted Pressure next 3h: 715.6


In [16]:
# dataset එක ලොකු නිසා sample එකක් ගන්න
df_model = df_model.sample(n=30000, random_state=42)

# date order එකට sort කරන්න
df_model = df_model.sort_values("datetime")

df_model.shape

ValueError: Cannot take a larger sample than population when 'replace=False'

In [17]:
sample_size = min(30000, len(df_model))

df_model = df_model.sample(n=sample_size, random_state=42)

df_model = df_model.sort_values("datetime")

print(df_model.shape)

(26006, 7)


In [18]:
sample_size = min(30000, len(df_model))

df_model = df_model.sample(n=sample_size, random_state=42)
df_model = df_model.sort_values("datetime")

df_model.shape

(26006, 7)

In [19]:
df_ne = df[df["Month"].isin([12, 1, 2])]
df_ne.head()

,Year,Month,Date,METAR /SPECI,Time(UTC),Wind Dir.,Wind speed(Kts),Visibility,Weather,Clouds,Dry tem(0C),Dew point(0C),RH(%),QNH (hPa),Unnamed: 14,time_clean,datetime
0,2019,1,1,M,10,30,7,9000,NaN,SKC,22.5,20.8,88,1015.2,NaN,0010,2019-01-01 00:10:00
1,2019,1,1,M,40,30,7,9000,NaN,SKC,22.6,20.5,88,1015.5,NaN,0040,2019-01-01 00:40:00
2,2019,1,1,M,110,30,8,8000,NaN,SKC,22.8,20.6,87,1015.6,NaN,0110,2019-01-01 01:10:00
3,2019,1,1,M,140,30,9,8000,NaN,SKC,23.1,20.7,86,1015.9,NaN,0140,2019-01-01 01:40:00
4,2019,1,1,M,210,30,11,9000,NaN,NSC,23.9,20.7,81,1016.3,NaN,0210,2019-01-01 02:10:00


In [20]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression MAE:", lr_mae)
print("Linear Regression R2:", lr_r2)

Linear Regression MAE: 2.562273053373192
Linear Regression R2: 0.5960549237317855


In [21]:
X = df_model[["temperature", "humidity", "pressure"]]
y = df_model[["temp_next", "humidity_next", "pressure_next"]]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [23]:
sample_size = min(30000, len(df_model))
df_model_sample = df_model.sample(n=sample_size, random_state=42)
df_model_sample = df_model_sample.sort_values("datetime")

X = df_model_sample[["temperature", "humidity", "pressure"]]
y = df_model_sample[["temp_next", "humidity_next", "pressure_next"]]

In [24]:

X = df_model[["temperature", "humidity", "pressure"]]

y = df_model[[
    "temp_next",
    "humidity_next",
    "pressure_next"
]]

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [26]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression MAE:", lr_mae)
print("Linear Regression R2:", lr_r2)

Linear Regression MAE: 2.562273053373192
Linear Regression R2: 0.5960549237317855


In [27]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest R2:", rf_r2)

Random Forest MAE: 2.22094450130635
Random Forest R2: 0.46761964924259386


In [28]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

X_train_lstm = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm_model = Sequential()
lstm_model.add(LSTM(50, input_shape=(1, 3)))
lstm_model.add(Dense(3))

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.fit(
    X_train_lstm,
    y_train,
    epochs=5,
    batch_size=32,
    verbose=1
)

lstm_pred = lstm_model.predict(X_test_lstm)

lstm_mae = mean_absolute_error(y_test, lstm_pred)
lstm_r2 = r2_score(y_test, lstm_pred)

print("LSTM MAE:", lstm_mae)
print("LSTM R2:", lstm_r2)

Epoch 1/5


C:\Users\imash\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 340100.0625
Epoch 2/5
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 333331.3438
Epoch 3/5
651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 326939.0000
Epoch 4/5
651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 320226.9062
Epoch 5/5
651/651 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 313651.8438
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
LSTM MAE: 334.5682373046875
LSTM R2: -75795.796875


In [29]:
ensemble_pred = (lr_pred + rf_pred + lstm_pred) / 3

ensemble_mae = mean_absolute_error(y_test, ensemble_pred)
ensemble_r2 = r2_score(y_test, ensemble_pred)

print("Hybrid Ensemble MAE:", ensemble_mae)
print("Hybrid Ensemble R2:", ensemble_r2)

Hybrid Ensemble MAE: 111.87957483364774
Hybrid Ensemble R2: -8418.554225803178


In [30]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Multiple Linear Regression",
        "Random Forest",
        "LSTM",
        "Hybrid Ensemble"
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        lstm_mae,
        ensemble_mae
    ],
    "R2 Score": [
        lr_r2,
        rf_r2,
        lstm_r2,
        ensemble_r2
    ]
})

results

,Model,MAE,R2 Score
0,Multiple Linear Regression,2.562273,0.596055
1,Random Forest,2.220945,0.467620
2,LSTM,334.568237,-75795.796875
3,Hybrid Ensemble,111.879575,-8418.554226


In [31]:
sample = pd.DataFrame([{
    "temperature": 28,
    "humidity": 80,
    "pressure": 1010
}])

lr_output = lr_model.predict(sample)
rf_output = rf_model.predict(sample)

sample_lstm = sample.values.reshape((sample.shape[0], 1, sample.shape[1]))
lstm_output = lstm_model.predict(sample_lstm)

final_output = (lr_output + rf_output + lstm_output) / 3

print("Predicted Temperature next 3h:", round(final_output[0][0], 2))
print("Predicted Humidity next 3h:", round(final_output[0][1], 2))
print("Predicted Pressure next 3h:", round(final_output[0][2], 2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
Predicted Temperature next 3h: 27.13
Predicted Humidity next 3h: 69.09
Predicted Pressure next 3h: 689.61


In [32]:
joblib.dump(lr_model, "linear_regression_model.pkl")
joblib.dump(rf_model, "random_forest_model.pkl")

lstm_model.save("lstm_model.h5")

print("All three models saved successfully")

All three models saved successfully


In [33]:
ensemble_pred = (lr_pred + rf_pred + lstm_pred) / 3

In [34]:
from sklearn.metrics import mean_absolute_error, r2_score

print("Hybrid MAE:", mean_absolute_error(y_test, ensemble_pred))
print("Hybrid R2:", r2_score(y_test, ensemble_pred))

Hybrid MAE: 111.87957483364774
Hybrid R2: -8418.554225803178


In [35]:
from sklearn.metrics import mean_absolute_error, r2_score

print("Hybrid MAE:", mean_absolute_error(y_test, ensemble_pred))
print("Hybrid R2:", r2_score(y_test, ensemble_pred))

Hybrid MAE: 111.87957483364774
Hybrid R2: -8418.554225803178


In [36]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import numpy as np

# load models
lr_model = joblib.load("linear_regression_model.pkl")
rf_model = joblib.load("random_forest_model.pkl")
lstm_model = load_model("lstm_model.h5")

# input
sample = pd.DataFrame([{
    "temperature": 28,
    "humidity": 80,
    "pressure": 1010
}])

# predictions
lr_out = lr_model.predict(sample)
rf_out = rf_model.predict(sample)

sample_lstm = sample.values.reshape((1,1,3))
lstm_out = lstm_model.predict(sample_lstm)

# final ensemble
final = (lr_out + rf_out + lstm_out) / 3

print("Temperature:", round(final[0][0],2))
print("Humidity:", round(final[0][1],2))
print("Pressure:", round(final[0][2],2))

ValueError: Could not deserialize 'keras.metrics.mse' because it is not a KerasSaveable subclass

In [37]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5", compile=False)

In [38]:
lstm_model.save("lstm_model.keras")

In [39]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.keras")

In [40]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

lr_model = joblib.load("linear_regression_model.pkl")
rf_model = joblib.load("random_forest_model.pkl")
lstm_model = load_model("lstm_model.h5", compile=False)

sample = pd.DataFrame([{
    "temperature": 28,
    "humidity": 80,
    "pressure": 1010
}])

lr_out = lr_model.predict(sample)
rf_out = rf_model.predict(sample)

sample_lstm = sample.values.reshape((1, 1, 3))
lstm_out = lstm_model.predict(sample_lstm)

final = (lr_out + rf_out + lstm_out) / 3

print("Temperature next 3h:", round(final[0][0], 2))
print("Humidity next 3h:", round(final[0][1], 2))
print("Pressure next 3h:", round(final[0][2], 2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
Temperature next 3h: 27.13
Humidity next 3h: 69.09
Pressure next 3h: 689.61
